# Build Model

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datetime import datetime as dt
import bentoml


pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)

In [2]:
from dota_oracle_common.postgresql import DatabaseEngineFactory
from sqlalchemy.ext.asyncio import AsyncSession

engine = DatabaseEngineFactory.get_engine()
engine

In [3]:
from dota_oracle_common.repositories.match_repository import MatchRepository


async with AsyncSession(engine) as session:
    match_repo = MatchRepository(session)
    
    matches = await match_repo.get_match_details(
        relationship_fields=[
            "outcome", "team_features", "player_hero_features", "hero_features"
        ]
    )
    
    print(f"number of matches: {len(matches)}")
    



dota_oracle_common.repositories.base_repository - Retrieved 96821 records for MatchTable
dota_oracle_common.repositories.match_repository - Found 96821 MatchTable details.


number of matches: 96821


In [4]:
match_outcome_list = []
hero_features_list = []
player_hero_team_features_list = []

for match in matches:
    outcome = match.outcome
    match_outcome_list.append(outcome.model_dump())
    
    team_features = match.team_features
    hero_features = match.hero_features
    player_hero_features = match.player_hero_features
    
    player_hero_team_features_dict = {**team_features.model_dump(), **player_hero_features.model_dump()}
    
    hero_features_list.append(hero_features.model_dump())
    player_hero_team_features_list.append(player_hero_team_features_dict)
    

print(f"count match_outcome_list: {len(match_outcome_list)}")
print(f"count features_list: {len(player_hero_team_features_list)}")

count match_outcome_list: 96821
count features_list: 96821


In [5]:
outcome_df = pd.DataFrame(match_outcome_list)
player_hero_team_df = pd.DataFrame(player_hero_team_features_list)
hero_df = pd.DataFrame(hero_features_list)

In [6]:
outcome_df

,match_id,radiant_win
0,8230722475,True
1,8230701740,False
2,8230693148,True
3,8230677659,True
4,8230656847,True
...,...,...
96816,5999283181,False
96817,5999249937,False
96818,5999214195,False
96819,5999201501,True


In [7]:
player_hero_team_df

,radiant_win_rate,dire_win_rate,match_id,radiant_dire_matchup,player_hero_0_win_rate,player_hero_3_win_rate,player_hero_128_win_rate,player_hero_130_win_rate,player_hero_132_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_4_win_rate,player_hero_129_win_rate,player_hero_131_win_rate
0,0.5,0.7,8230722475,0.333333,0.600000,0.75,0.800000,0.714286,0.80,0.550000,0.600000,0.4,0.769231,0.500000
1,0.7,0.5,8230701740,0.600000,0.642857,0.70,0.300000,0.300000,0.50,0.500000,0.384615,0.4,0.666667,0.550000
2,0.6,0.6,8230693148,0.625000,0.777778,1.00,0.588235,0.600000,0.50,0.714286,0.692308,0.5,0.500000,0.833333
3,0.4,0.8,8230677659,0.350000,0.250000,0.45,0.625000,1.000000,0.00,0.350000,0.625000,0.6,0.600000,1.000000
4,0.7,0.4,8230656847,0.600000,0.450000,0.40,0.384615,0.500000,0.65,0.538462,0.400000,0.5,0.350000,0.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,0.5,0.5,5999283181,0.500000,0.500000,0.50,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96817,1.0,0.0,5999249937,1.000000,0.500000,1.00,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.000000,0.500000
96818,0.5,0.5,5999214195,0.500000,0.500000,0.50,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96819,0.5,0.5,5999201501,0.500000,0.500000,0.50,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000


In [8]:
hero_df

,match_id,hero_picks
0,8230722475,"[18, 19, 51, 79, 55, 34, 49, 95, 64, 102]"
1,8230701740,"[68, 97, 14, 13, 18, 102, 42, 74, 11, 98]"
2,8230693148,"[34, 120, 19, 52, 54, 49, 106, 68, 41, 119]"
3,8230677659,"[22, 14, 42, 35, 43, 75, 39, 60, 19, 63]"
4,8230656847,"[64, 29, 25, 86, 19, 1, 14, 16, 11, 5]"
...,...,...
96816,5999283181,"[58, 98, 19, 42, 111, 41, 68, 96, 128, 46]"
96817,5999249937,"[126, 78, 109, 128, 102, 86, 96, 106, 94, 68]"
96818,5999214195,"[96, 123, 15, 121, 72, 23, 2, 94, 128, 30]"
96819,5999201501,"[68, 58, 97, 106, 19, 111, 17, 61, 88, 6]"


In [11]:
from dota_oracle_pipeline.feature_transformation.feature_encoder import FeatureEncoder
from dota_oracle_common.repositories.heroes_repository import HeroesRepository

async with AsyncSession(engine) as session:
    heros_repo = HeroesRepository(session)
    hero_map = await heros_repo.get_hero_id_map()

encoded_hero_features = FeatureEncoder.encode_hero_features(hero_features=hero_df, hero_map=hero_map)

In [12]:
encoded_hero_features

,Pugna,Anti-Mage,Axe,Bane,Bloodseeker,Crystal Maiden,Drow Ranger,Earthshaker,Juggernaut,Mirana,Morphling,Shadow Fiend,Phantom Lancer,Puck,Pudge,Razor,Sand King,Storm Spirit,Sven,Tiny,Vengeful Spirit,Windranger,Zeus,Kunkka,Lina,Lion,Shadow Shaman,Slardar,Tidehunter,Witch Doctor,Lich,Riki,Enigma,Tinker,Sniper,Necrophos,Warlock,Beastmaster,Queen of Pain,Venomancer,Faceless Void,Wraith King,Death Prophet,Phantom Assassin,Templar Assassin,Viper,Luna,Dragon Knight,Dazzle,Clockwerk,...,Shadow Demon,Lone Druid,Chaos Knight,Meepo,Treant Protector,Ogre Magi,Undying,Rubick,Disruptor,Nyx Assassin,Naga Siren,Keeper of the Light,Io,Visage,Slark,Medusa,Troll Warlord,Centaur Warrunner,Magnus,Timbersaw,Bristleback,Tusk,Skywrath Mage,Abaddon,Elder Titan,Legion Commander,Techies,Ember Spirit,Earth Spirit,Underlord,Terrorblade,Phoenix,Oracle,Winter Wyvern,Arc Warden,Monkey King,Dark Willow,Pangolier,Grimstroke,Hoodwink,Void Spirit,Snapfire,Mars,Ring Master,Dawnbreaker,Marci,Primal Beast,Muerta,Kez,match_id
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230722475
1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230701740
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,8230693148
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230677659
4,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230656847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,5999283181
96817,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,5999249937
96818,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,5999214195
96819,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5999201501


In [ ]:
merged_df = pd.merge(outcome_df, (pd.merge(player_hero_team_df, encoded_hero_features, how='inner')), how='inner')
merged_df

In [ ]:
df_final = merged_df.drop('match_id', axis=1)

In [ ]:
# Train Test Split
X = df_final.drop('radiant_win', axis=1)
y = df_final['radiant_win']

n_total = len(df_final)
n_test = int(n_total * 0.3)

# Top 30% as test, remaining 70% as train
X_test = X.iloc[:n_test]     # First 30%
X_train = X.iloc[n_test:]    # Remaining 70%
y_test = y.iloc[:n_test]     # First 30%
y_train = y.iloc[n_test:]    # Remaining 70

In [ ]:
# Import candidate classifiers

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# import metrics

from sklearn.metrics import accuracy_score


In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000  # Increase if convergence issues
    ),
    
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # Use all cores
    ),
    
    'XGBoost': xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',  # Suppress warning
        n_estimators=100
    ),
    
    'LightGBM': lgb.LGBMClassifier(
        random_state=42,
        verbosity=-1,  # Suppress output
        n_estimators=100
    )
}

In [ ]:
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]  # For ROC-AUC
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name} Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

In [ ]:
clf = models['Random Forest']
clf

In [ ]:
# Prediction with bentoml model service

In [ ]:
from dota_oracle_common.models.inference.schema import VersionMetaData, PerformanceMetrics, ModelMetaData 

def save_sklearn_model(
    model,
    model_name: str,
    metadata: ModelMetaData
):
    print(f"attempting to save model {model_name}")
    saved_model = bentoml.sklearn.save_model(
        name=model_name,
        model=model, 
        signatures={'predict':{'batchable':True}},
        metadata=metadata.model_dump()
    )
    
    print(f"Model saved: {saved_model}")

In [ ]:
model_name = 'dota_oracle_random_forest'

feature_columns = clf.feature_names_in_.tolist()

performance = PerformanceMetrics(
    accuracy=0.551
)

version_data = VersionMetaData(
    performance_metrics=performance,
    feature_columns=feature_columns
)

metadata = metadata = ModelMetaData(
        name=model_name,
        version='0.0.1',
        trained_date=dt.now(),
        version_metadata=version_data
)

try:
    save_sklearn_model(model=clf, model_name=model_name, metadata=metadata)
    print("process complete")
except Exception as e:
    print(f"error saving model to bentoml, {e}")